# Iris — Probabilistic Generative Classifier (full web UI in Colab)

A hand-written NumPy **Gaussian generative classifier** (MLE means/covariances, Bayes posteriors, log-space discriminants) with a **FastAPI backend** and a **WebGL front end** where a 3D iris reshapes itself from your measurements.

**Run order:** just *Runtime → Run all*. The whole codebase (`src/`, `backend/`, `web/`, `data/`, `tests/`) is embedded in cell 2, so nothing needs uploading.


In [ ]:
# 1. Dependencies (numpy / pandas / scikit-learn already ship with Colab)
!pip -q install fastapi "uvicorn>=0.29" pydantic

In [ ]:
# 2. Unpack the project into /content/iris  (identical to the local codebase)
import base64, io, os, zipfile, shutil
PROJECT_ZIP_B64 = "..."
if os.path.exists("/content/iris"): shutil.rmtree("/content/iris")
os.makedirs("/content/iris")
zipfile.ZipFile(io.BytesIO(base64.b64decode(PROJECT_ZIP_B64))).extractall("/content/iris")
os.chdir("/content/iris")
print(sorted(os.listdir(".")))
print("src:", sorted(os.listdir("src")))


In [ ]:
# 3. The model on its own - no UI, just the ML result
import sys; sys.path.insert(0, "/content/iris")
from src.data_loader import load_and_prepare_data
from src.generative_classifier import GaussianGenerativeClassifier
from src.evaluation import evaluate_model

X_tr, X_te, y_tr, y_te, feats, targets = load_and_prepare_data(save_csv=False)
clf = GaussianGenerativeClassifier().fit(X_tr, y_tr)
res = evaluate_model(clf, X_te, y_te, targets)
print(f"Test accuracy: {res['accuracy']*100:.2f}%")
print()
print(res["classification_report"])
print("Confusion matrix:")
print(res["confusion_matrix"])


In [ ]:
# 4. Optional: run the project's unit tests
!pip -q install pytest
!cd /content/iris && python -m pytest -q tests 2>&1 | tail -5

In [ ]:
# 5. Start the FastAPI backend (serves /api/* and the web UI) in a background thread
import threading, time, socket, urllib.request, uvicorn

PORT = 8000
def up():
    s = socket.socket(); s.settimeout(0.3)
    try: return s.connect_ex(("127.0.0.1", PORT)) == 0
    finally: s.close()

if not up():
    from backend.server import app
    server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="warning"))
    threading.Thread(target=server.run, daemon=True).start()
    for _ in range(50):
        if up(): break
        time.sleep(0.2)
print("server up:", up())
print(urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/meta").read()[:90], "...")


In [ ]:
# 6. Show the UI (Colab proxies the port to your browser)
from google.colab import output
output.serve_kernel_port_as_iframe(8000, height=900)

### Notes
* The UI loads Three.js, KaTeX and Google Fonts from CDNs, so the Colab session needs internet (it has it by default).
* If the frame looks cramped, widen it with the little *open in new tab* arrow at the frame's top-right, or re-run cell 6 with a larger `height`.
* The classifier itself is the unmodified `src/generative_classifier.py` — the UI only talks to it through `backend/server.py`.
